# Earthquake or nuclear explosion?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/09_explosion_or_earthquake.ipynb).

The Comprehensive Nuclear-Test-Ban Treaty, opened for signature in 1996, bans every
nuclear explosion, though it has not yet entered into force. A test can be hidden underground, out
of sight, but the ground still shakes: an explosion shakes it like an earthquake, and stations
thousands of kilometres away record it. (Firing a device inside a large underground cavity can
muffle the shaking many times over, one reason small tests are hard to find.) The trouble is that
the Earth has thousands of earthquakes a year. To
check a ban, you have to pick out the explosions among them, and you have to be right, because a
false alarm accuses a country of breaking a treaty.

Today you work with 1,118 real events from the International Seismological Centre's
catalogue: 117 nuclear explosions and 1,001 earthquakes in the regions
around the test sites. You score the simplest rules first, find what each one misses, and then fit a line that
gives every event a probability. At the end of class you ask the line about five explosions it
has never seen, North Korea's. At home you fit the line in one part of the world and check it in
another.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say why an explosion and an earthquake of the same size shake the ground
differently, and which seismic measurement shows it; and judge a rule that flags explosions by
the two mistakes it can make.

**The code.** `accuracy_score`, `precision_score`, `recall_score`, `f1_score` and
`confusion_matrix` · `train_test_split` with `stratify` · `LogisticRegression` with `fit`,
`predict` and `predict_proba` · an ROC curve with `roc_auc_score`.

**Twelve places where you write something: eight in class, four at home.** Each is headed
*Your turn*, and the cell under it has the shape of the answer written in, with `...` wherever
your code goes and a comment beside each one saying what goes there.

The four questions of the class:

1. What counts as right when explosions are rare?
2. Can one magnitude tell an explosion from an earthquake?
3. Can two magnitudes together tell?
4. Where should the alarm go, and would it catch North Korea's explosions?

## Setup

Run the next cell once. It reads the events, stored with the course: every event in three regions
with test sites, plus North Korea, for which the International Seismological Centre (ISC) Bulletin
lists both magnitudes this week uses. It splits off the five North Korean explosions into their own table, `korea`, for the end of
class, and keeps the rest in `events`. It also reads the coastlines for the map, and makes `grid`,
a table of points spread over the mb–Ms plane, used for the probability maps at the end of class
and in the homework.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# One row per event: when and where, its two magnitudes, and what it was.
# Source: International Seismological Centre (2026), On-line Bulletin, https://doi.org/10.31905/D808B830
table = pd.read_csv(CACHE + "/week09_mb_ms_isc.csv")
events = table[table["region"] != "Korea"]                           # the class's events
korea = table[(table["region"] == "Korea") & table["is_explosion"]]  # North Korea's explosions, kept apart
coast = pd.read_csv(CACHE + "/coastlines.csv")

# the grid: every pair of mb from 3.5 to 7 and Ms from 2.5 to 8, in steps of 0.05, for the maps
grid_mb = []
grid_Ms = []
for mb in np.arange(3.5, 7.01, 0.05):
    for Ms in np.arange(2.5, 8.01, 0.05):
        grid_mb.append(mb)
        grid_Ms.append(Ms)
grid = pd.DataFrame({"mb": grid_mb, "Ms": grid_Ms})
print(len(events), "events:", events["is_explosion"].sum(), "explosions and", (events["is_explosion"] == False).sum(), "earthquakes")

## 1. What counts as right when explosions are rare?

**An underground test.** A nuclear device is placed hundreds of metres underground, at the bottom
of a drilled shaft or at the end of a tunnel dug into a mountain, sealed in, and fired. Usually nothing reaches the surface but the
shaking (and sometimes radioactive gas that seeps out), and later, often, a crater where the rock above the blast collapses.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week09/underground_test_schematic.png" width="420"> <img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week09/nevada_subsidence_craters.jpg" width="380">

*Left: a vertical underground test at the Nevada Test Site, the device at the bottom of a drilled
hole. Diagram: US Government (public domain). Right: craters left at the Nevada Test Site where
the ground collapsed into the cavities of underground tests. Photo: USGS (public domain). Both via
Wikimedia Commons.*

**Two sizes for one event.** Shaking travels as waves. The fastest, the P waves, push and pull the
rock along their path; S waves shake it from side to side; and the slow surface waves roll along
the ground behind them.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week09/seismic_waves.jpg" width="300">

*How each kind of seismic wave moves the ground: two body waves (P and S) and two surface waves
(Love and Rayleigh). Figure: USGS (public domain), via Wikimedia Commons.*

An explosion pushes the rock outward from one small spot, the same in every direction, so most of
its shaking leaves as P waves. An earthquake is two blocks of rock slipping past each other along a
fault: it sends out strong S waves, and, because the slip spreads over a large patch of fault and
lasts longer, strong surface waves too.

**What a seismometer records.**

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week09/north_korea_2006_seismograms.jpg" width="640">

*North Korea's first test, 9 October 2006. Left: the test site (red star), earthquakes nearby (blue
dots) and two seismic stations (triangles). Right: the explosion (red) and a recent earthquake
nearby (blue) recorded at a station: the explosion's shaking is mostly P waves; the earthquake's
largest shaking is its S waves. Figure: Lawrence Livermore National Laboratory, Science &
Technology Review (March 2009), via the Berkeley Seismological Laboratory blog.*

So seismologists give each event two sizes. The **body-wave magnitude mb** is read from the first,
fast P wave, from waves that shake about once a second. The **surface-wave magnitude Ms** is read
from the slow surface waves that follow, about twenty seconds long. A small, sudden source like an
explosion puts little energy into such long waves, so for its mb an explosion has a small Ms. The
table holds both, `mb` and `Ms`, for every event.

In [ ]:
events[["date", "region", "depth", "mb", "Ms", "is_explosion"]].head()

Each row is one event: its date, its region, its depth below the surface in km, its two
magnitudes, and `is_explosion`, the **label**: True for a nuclear explosion, False for an
earthquake. The events come from three regions with test sites, from 1970 to 1996:

- the western US: 405 events, 36 of them
  explosions, 35 at the Nevada Test Site and one in Colorado;
- Central Asia: 696 events, 64 explosions,
  53 at the Soviet test site near Semipalatinsk (now in Kazakhstan) and
  11 at China's Lop Nur;
- Novaya Zemlya, an Arctic island where the Soviet Union also tested: 17
  events, all explosions.

Only events with both magnitudes are kept. Small explosions often have no Ms at all, so the table
holds mostly large ones: only 9 of its 117 explosions have an
mb below 5.4. Keep that in mind when the model meets smaller ones.

In [ ]:
explosions = events[events["is_explosion"]]
earthquakes = events[events["is_explosion"] == False]

plt.figure(figsize=(10, 4.5))
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.5)
plt.scatter(earthquakes["lon"], earthquakes["lat"], s=6, color="tab:blue", label="earthquake")
plt.scatter(explosions["lon"], explosions["lat"], s=30, marker="*", color="red", label="nuclear explosion")
plt.xlim(-180, 180)
plt.ylim(0, 90)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("The events of the table")
plt.legend(loc="lower left")
plt.show()

**This is classification.** In the regression weeks a model answered with a number. Here it
answers with a label: explosion or earthquake. A **guess** is a column of True and False, one per
event, from a rule or a model. It is scored against the label.

The laziest rule is to say "earthquake" every time. It is the **baseline**, the simplest rule
there is, scored first: a model that cannot beat it does no useful work.

### ✏️ Your turn 1

**How good is "always earthquake"?**

**Why:** a score is only useful if a rule that does no work scores badly. Find out what the usual
score, accuracy, gives the laziest rule.

1. **`always_earthquake`**: a guess of False for every event, given. It is written as a mask on
   a column (no event has an mb above 100) so that it is a column of True and False like every
   other guess this week.
2. **`accuracy`**: the share of events where the guess equals the label. Work it out by hand as the
   mean of the mask `always_earthquake == events["is_explosion"]`, then check it with
   `accuracy_score(label, guess)`.
3. **`found`**: how many explosions the rule flags: the sum of the mask `always_earthquake &
   events["is_explosion"]`.

**Use these names**, because the self-check looks for them: `accuracy` and `found`.

In [ ]:
# ← your answer here: replace every ... with code
always_earthquake = events["mb"] > 100  # False for every event
accuracy = (... == ...).mean()          # the guess, and the label
print("accuracy by hand:", round(accuracy, 3), "  with accuracy_score:", round(accuracy_score(events["is_explosion"], always_earthquake), 3))
found = (... & ...).sum()               # flagged, and an explosion
print("explosions flagged:", found, "of", events["is_explosion"].sum())

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(accuracy, 3) == 0.895, "the mean of the mask always_earthquake == events['is_explosion']"
assert found == 0, "the sum of always_earthquake & events['is_explosion']"
print(f"✓ Your turn 1 — 'always earthquake' scores {accuracy:.3f} and finds {found} explosions")

**Accuracy 0.895, and not one explosion found.** Nine events in ten are
earthquakes, so a rule that never flags is right nine times in ten. When one label is rare,
accuracy hides the misses. A treaty needs scores that count the two kinds of mistake apart.

## 2. Can one magnitude tell an explosion from an earthquake?

**Four counts.** Sort the events by label and by guess into four groups. A **hit** is an explosion
the rule flagged, a **miss** an explosion it did not flag, and a **false alarm** an earthquake it
flagged. The fourth group, earthquakes the rule leaves alone, is the right answer for most events. Two scores
come from the counts:

- **precision** = hits / (hits + false alarms): of the events you flagged, the share that were
  explosions;
- **recall** = hits / (hits + misses): of the explosions, the share you flagged.

In plain words, precision asks: of the ones you flagged, how many were right? Recall asks: of the
real ones, how many did you catch? **F1** joins them into one number,
$F_1 = 2PR/(P+R)$ with $P$ the precision and $R$ the recall, which is high only when both are
high.

Ten made-up events, two of them explosions, and the rule "flag every event with mb 5.7 or more":

In [ ]:
ten_events = pd.DataFrame({"mb": [4.2, 4.5, 4.8, 5.0, 5.1, 5.3, 5.6, 5.8, 5.9, 6.2],
                           "is_explosion": [False, False, False, False, False, True, False, False, True, False]})
ten_events["flagged"] = ten_events["mb"] >= 5.7
print(ten_events)
print("confusion_matrix:")
print(confusion_matrix(ten_events["is_explosion"], ten_events["flagged"]))

Count by eye: the rule flags the three events with mb 5.8, 5.9 and 6.2. One of them, 5.9, is an
explosion: 1 hit and 2 false alarms. The explosion with mb 5.3 is missed. The six other earthquakes
are left alone. `confusion_matrix` prints the same four counts as a table of 2 by 2: rows are the
label (False, then True), columns the guess (False, then True), so

    [[left alone, false alarms],
     [misses,     hits        ]]

Precision is 1 / (1 + 2) = 1/3, recall 1 / (1 + 1) = 1/2, and F1 = 2 × 1/3 × 1/2 / (1/3 + 1/2) = 0.4.

### ✏️ Your turn 2

**How well does a cut on mb alone find the explosions?**

**Why:** the simplest rule that looks at the data is a cut on one column. It is the rule a model
must beat, and scoring it uses all three new scores.

1. **`guess_mb`**: True where an event's mb is 5.7 or more.
2. **`hits`, `misses`, `false_alarms`**: the three counts, each as the sum of a mask. A hit is
   flagged and an explosion; a miss is not flagged and an explosion; a false alarm is flagged and
   an earthquake. (`guess_mb == False` is the mask of the events not flagged.)
3. **`f1_mb`**: `f1_score(label, guess)`. The precision and recall are printed beside it.
4. **The figure**: histograms of mb for the earthquakes and the explosions, with a line at the cut.

**Use these names**, because the self-check looks for them: `hits`, `misses`, `false_alarms` and
`f1_mb`.

In [ ]:
# ← your answer here: replace every ... with code
label = events["is_explosion"]
guess_mb = events["mb"] >= ...                    # the cut
hits = (... & ...).sum()                          # flagged, and an explosion
misses = ((guess_mb == False) & ...).sum()        # not flagged, and an explosion
false_alarms = (guess_mb & (... == False)).sum()  # flagged, and an earthquake
f1_mb = f1_score(..., ...)                        # the label first, then the guess
print(hits, "hits,", misses, "misses,", false_alarms, "false alarms")
print("precision", round(precision_score(label, guess_mb), 3), " recall", round(recall_score(label, guess_mb), 3), " F1", round(f1_mb, 3))

plt.figure(figsize=(8, 4))
bins = np.arange(3.65, 7.05, 0.1)                                     # one bin per tenth of a magnitude, the same for both
plt.hist(..., bins=bins, alpha=0.6, label="earthquakes")              # the earthquakes' mb
plt.hist(..., bins=bins, alpha=0.6, color="red", label="explosions")  # the explosions' mb
plt.axvline(..., color="black", label="the cut")                      # the cut
plt.xlabel("body-wave magnitude mb")
plt.ylabel("number of events")
plt.title("mb alone: the two labels overlap")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (hits, misses, false_alarms) == (85, 32, 105), "a hit is guess & label; a miss (guess == False) & label; a false alarm guess & (label == False)"
assert round(f1_mb, 3) == 0.554, "f1_score(label, guess_mb)"
print(f"✓ Your turn 2 — the cut on mb: {hits} hits, {misses} misses, {false_alarms} false alarms, F1 {f1_mb:.3f}")

The cut finds 85 of 117 explosions, but it also flags
105 earthquakes: precision 0.447, recall 0.726, F1
0.554. The histogram shows why no cut on mb can do much better: big earthquakes and big
explosions have the same mb. Ms alone is worse:

In [ ]:
guess_ms = events["Ms"] <= 4.9     # flag events with small surface waves; 4.9 is about the best cut on Ms
print("Ms of 4.9 or less: F1", round(f1_score(label, guess_ms), 3))

F1 0.231. Each magnitude alone fails, because in each one the two labels overlap. Before
trying two columns together, one more column looks tempting: depth. Earthquakes start kilometres
down; a test hole is a few hundred metres deep.

### ✏️ Your turn 3

**How well does "depth is 0" find the explosions?**

**Why:** a rule that scores almost perfectly deserves the hardest look.

1. **`guess_depth`**: True where an event's `depth` is exactly 0.
2. **`f1_depth`**: its F1.

**Use these names**, because the self-check looks for them: `guess_depth` and `f1_depth`.

In [ ]:
# ← your answer here: replace every ... with code
guess_depth = events["depth"] == ...  # a depth of exactly 0
f1_depth = f1_score(..., ...)         # the label, then the depth guess
print("depth is 0: F1", round(f1_depth, 3))
print("explosions with depth 0:", (guess_depth & label).sum(), "of", label.sum(), "  earthquakes with depth 0:", (guess_depth & (label == False)).sum())

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(f1_depth, 3) == 0.996, "guess_depth is events['depth'] == 0; then f1_score"
print(f"✓ Your turn 3 — 'depth is 0' scores F1 {f1_depth:.3f}")

**An F1 of 0.996, nearly perfect, and useless here.** Every explosion in the table has
depth 0.0, and the catalogue marks each of those depths as fixed by the analyst, not measured from
the waves. The analyst set it to 0 because the event was already known to be an explosion. The
column knew the answer before anyone looked. This is called **leakage**: a column that already
holds the answer. When a score looks too good, 99 percent say, first ask whether one of the columns
already knew the answer.

A depth measured from the waves is still useful in real monitoring: an event clearly tens of
kilometres deep is no test. But shallow depths are hard to measure (the catalogue fixed the depth
of 343 of these earthquakes too), so depth cannot separate two shallow events.

## 3. Can two magnitudes together tell?

An explosion pushes the rock outward from one small spot: it makes strong P waves and weak surface
waves. An earthquake slips along a fault and makes strong surface waves. So at the same mb, an
explosion should have the smaller Ms. Plot the two together:

In [ ]:
plt.figure(figsize=(6.5, 5.5))
plt.scatter(earthquakes["mb"], earthquakes["Ms"], s=8, color="tab:blue", alpha=0.5, label="earthquake")
plt.scatter(explosions["mb"], explosions["Ms"], s=14, color="red", alpha=0.7, label="explosion")
plt.plot([3.5, 7], [3.5, 7], color="0.4", label="Ms = mb")
plt.text(3.6, 7.3, "magnitudes have one decimal, so many events sit on the same point")
plt.xlabel("body-wave magnitude mb")
plt.ylabel("surface-wave magnitude Ms")
plt.title("Two magnitudes together")
plt.legend()
plt.show()

The two labels, which overlapped in each column, come apart in the two together: the explosions
sit below the earthquakes, along a line. A line below Ms = mb, by some gap, could split them:
flag an event when Ms < mb − gap.

Magnitudes have one decimal, so many events sit exactly on a gap like 1.0, and rounding inside the
computer would decide their side. So try gaps that end in 5, which no event sits on: 0.55, 0.75,
0.95, 1.15, 1.35.

### ✏️ Your turn 4

**Where would you draw the line by eye?**

**Why:** a line by eye is the baseline for a fitted line. If fitting cannot beat it, fitting is
decoration.

1. **The loop, which you write yourself.** For each gap in `gaps`: make the guess, True where Ms is
   below mb minus the gap (as `guess_mb` in Your turn 2, with the line in place of the cut); score
   it with `f1_score`, the label first; and print the gap and its F1. Each comment in the cell is
   one line for you to write. It is the shape of the loop in week 2: `for x in things:`, then the
   lines that repeat, indented.
2. **`gap`**: the gap with the best F1, read from your printout.
3. **The figure**: the scatter again, with your line, Ms = mb − gap. The lines that score your
   chosen gap, `guess_line` and `f1_line`, are given.

**Use these names**, because the self-check looks for them: `gap` and `f1_line`.

In [ ]:
# ← your answer here: replace every ... with code
gaps = [0.55, 0.75, 0.95, 1.15, 1.35]
# for each gap in gaps:
#     guess: True where events["Ms"] is below events["mb"] minus the gap
#     f1_gap: the F1 of that guess, the label first
#     print the gap and round(f1_gap, 3)
...

gap = ...  # the gap with the best F1 in your printout
guess_line = events["Ms"] < events["mb"] - gap
f1_line = f1_score(label, guess_line)
print("chosen gap", gap, ": F1", round(f1_line, 3), "with", (guess_line & label).sum(), "hits and", (guess_line & (label == False)).sum(), "false alarms")

mb_ends = np.array([3.5, 7])                                           # the range of mb in the table, to draw the line across
plt.figure(figsize=(6.5, 5.5))
plt.scatter(earthquakes["mb"], earthquakes["Ms"], s=8, color="tab:blue", alpha=0.5, label="earthquake")
plt.scatter(explosions["mb"], explosions["Ms"], s=14, color="red", alpha=0.7, label="explosion")
plt.plot(mb_ends, ..., color="black", label="my line: Ms = mb - gap")  # the line's Ms at the two ends
plt.xlabel("body-wave magnitude mb")
plt.ylabel("surface-wave magnitude Ms")
plt.title("A line by eye")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert gap == 0.95 and round(f1_line, 3) == 0.831, "of the five gaps, 0.95 gives the best F1; guess_line is Ms < mb - gap"
print(f"✓ Your turn 4 — the line by eye: gap {gap}, F1 {f1_line:.3f}")

With a gap of 0.95 the line flags 106 of 117 explosions and
32 earthquakes: F1 0.831, against 0.554 for mb alone. Your
loop printed 0.499, 0.691, 0.831, 0.817 and 0.587 for
the five gaps.

**But this line was chosen by looking at every event,** and then scored on the same events. It is
the mistake the model-selection week warned about: hide some data from yourself, then check. And its slope
was fixed at 1 by hand. A fitted line can choose its own slope, and it can be checked on events it
never saw.

**Logistic regression** is the straight line of the regression weeks, but its answer is a
probability between 0 and 1. It works out a value $z$ from the columns, as `LinearRegression` did,
and then bends $z$ into a probability with the S-curve

$$p = \frac{1}{1 + e^{-z}}, \qquad z = a \times \text{mb} + b \times \text{Ms} + c$$

where $p$ is the probability that the event is an explosion, $a$, $b$, $c$ are the three numbers
`fit` finds, and $e \approx 2.718$. For $z = -2$, $p = 0.12$; for $z = 0$, $p = 0.5$; for $z = 2$,
$p = 0.88$: the larger $z$, the nearer $p$ is to 1. The events where $z = 0$ form a straight line
on the mb–Ms plane, $a \times \text{mb} + b \times \text{Ms} + c = 0$, or, solved for Ms,
$\text{Ms} = -(a \times \text{mb} + c)/b$. That line is the **decision boundary**, where the
model's answer flips from one label to the other; for logistic regression it is always a straight
line. On one side the model says explosion, on the other earthquake.

Six made-up events with one number each, the first three earthquakes and the last three
explosions:

In [ ]:
six_events = pd.DataFrame({"size": [1, 2, 3, 4, 5, 6],
                           "is_explosion": [False, False, False, True, True, True]})
six_model = LogisticRegression().fit(six_events[["size"]], six_events["is_explosion"])
six_events["probability"] = six_model.predict_proba(six_events[["size"]])[:, 1].round(2)
print(six_events)

sizes = pd.DataFrame({"size": np.arange(0, 7.05, 0.05)})     # sizes from 0 to 7, for drawing the curve
plt.figure(figsize=(6, 3.5))
plt.plot(sizes["size"], six_model.predict_proba(sizes)[:, 1], color="black")
plt.scatter(six_events["size"], six_events["is_explosion"], color="red")
plt.axhline(0.5, color="0.6")
plt.xlabel("size")
plt.ylabel("probability of an explosion")
plt.title("The S-curve on six made-up events")
plt.show()

The probabilities rise from 0.06 to 0.94, and the curve crosses 0.5 halfway between
the last earthquake and the first explosion, where the eye would put it. `predict_proba(...)[:, 1]`
gives the probability of True, an explosion, for each row; `predict` flags the rows where it is
above 0.5.

**How `fit` chooses the curve.** Give each event the probability the curve gives to the label it
really has: $p$ for an explosion, $1 - p$ for an earthquake. A good curve gives every event a high
probability for its own label. Multiply the six together: that is the chance the curve gives to
all six labels coming out as they did. `fit` tries values of the numbers and keeps the ones that
make this chance as large as it can (while keeping the numbers from growing huge). Check it
against the same curve moved one unit to the right, which crosses 0.5 at size 4.5 instead of 3.5:

In [ ]:
w = six_model.coef_[0][0]      # the number fit found for size
c = six_model.intercept_[0]    # and the constant
size = six_events["size"]
six_label = six_events["is_explosion"]

for name, z in [("fitted curve", w * size + c), ("curve moved right by 1", w * (size - 1) + c)]:
    p = 1 / (1 + np.exp(-z))                          # the S-curve
    own = six_label * p + (1 - six_label) * (1 - p)   # p for an explosion, 1 - p for an earthquake
    chance = 1
    for q in own:
        chance = chance * q                           # multiply the six together
    print(name, ": probability of each event's own label", list(own.round(2)), " all six:", round(chance, 3))

The moved curve gives the event of size 4, an explosion, only 0.36: it
says "earthquake" about an explosion, and the chance of all six falls from 0.256 to
0.152. The fitted curve scores higher. (On six events this cleanly split, an ever steeper
curve would score higher still; `fit` stops that by keeping the numbers small.) With two columns,
`fit` does the same with $a$, $b$ and $c$.

**Hold out events the model never sees.** `train_test_split` cuts the rows at random into a
training part, which `fit` sees, and a held-out part, which only the score sees. With few
explosions a random cut can leave too few in one part, so `stratify=events["is_explosion"]` keeps
the share of explosions the same in both.

### ✏️ Your turn 5

**Does a fitted line beat the line by eye on events it never saw?**

**Why:** the line by eye was drawn on every event and scored on the same ones. The fitted line
learns from the training events only, so its score on the held-out events is fair.

1. **`train`, `held_out`**: `train_test_split(events, test_size=0.3, random_state=42,
   stratify=events["is_explosion"])`, given.
2. **`model`**: `LogisticRegression().fit(...)` on the training events: the two columns `mb` and
   `Ms`, then the label.
3. **`guess_model`**: `model.predict(...)` on the held-out events' two columns.
4. **`f1_model`**: its F1 on the held-out events; and **`f1_eye`**: the line by eye, gap 0.95, on
   the same held-out events.

**Use these names**, because the self-check looks for them: `model`, `f1_model` and `f1_eye`.

In [ ]:
# ← your answer here: replace every ... with code
train, held_out = train_test_split(events, test_size=0.3, random_state=42, stratify=events["is_explosion"])
print(len(train), "training events (", train["is_explosion"].sum(), "explosions );", len(held_out), "held out (", held_out["is_explosion"].sum(), ")")

model = LogisticRegression().fit(..., ...)         # the training events: the mb and Ms columns, then the label
guess_model = model.predict(held_out[[..., ...]])  # the held-out events' two magnitudes
f1_model = f1_score(..., ...)                      # the held-out label, then the model's guess
guess_eye = held_out["Ms"] < held_out["mb"] - 0.95
f1_eye = f1_score(..., ...)                        # the held-out label, then the eye line's guess
print("held-out F1: fitted line", round(f1_model, 3), "  line by eye", round(f1_eye, 3))
print(confusion_matrix(held_out["is_explosion"], guess_model))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(f1_model, 3) == 0.923, "fit on train[['mb', 'Ms']] and train['is_explosion'], predict held_out[['mb', 'Ms']]"
assert round(f1_eye, 3) == 0.849, "the line by eye scored on the held-out events only"
print(f"✓ Your turn 5 — held-out F1: fitted line {f1_model:.3f}, line by eye {f1_eye:.3f}")

In [ ]:
a, b = model.coef_[0]          # the numbers fit found for mb and Ms
c = model.intercept_[0]        # and the constant
print("the boundary: Ms =", round(-a / b, 2), "x mb", round(-c / b, 2))
mb_ends = np.array([3.5, 7])     # the range of mb in the table
Ms_boundary = -(a * mb_ends + c) / b     # where z = 0, solved for Ms
held_explosions = held_out[held_out["is_explosion"]]
held_earthquakes = held_out[held_out["is_explosion"] == False]

plt.figure(figsize=(6.5, 5.5))
plt.scatter(held_earthquakes["mb"], held_earthquakes["Ms"], s=10, color="tab:blue", label="earthquake")
plt.scatter(held_explosions["mb"], held_explosions["Ms"], s=18, color="red", label="explosion")
plt.plot(mb_ends, Ms_boundary, color="black", label="model's boundary: probability 0.5")
plt.plot(mb_ends, mb_ends - 0.95, color="black", ls="--", label="line by eye")
plt.xlabel("body-wave magnitude mb")
plt.ylabel("surface-wave magnitude Ms")
plt.title(f"Boundary fitted on the training events, drawn on the {len(held_out)} held-out ones")
plt.legend()
plt.show()

On the 336 held-out events (35 explosions), the fitted line flags
30 explosions and no earthquake: F1 0.923. The line by eye flags
31 explosions and 7 earthquakes: F1
0.849. On events like the ones it learned from, fitting bought something. The model's
boundary is also steeper, Ms = 1.72 × mb − 5.1: the data chose a slope of
1.72, not 1. Look at where the two lines part. At small mb the boundary falls well below the
line by eye, so a small explosion needs an unusually small Ms to be flagged. Only
9 of the 117 explosions have an mb below 5.4, so the slope was
set by the large ones. Keep that in mind for North Korea.

It still misses 5 explosions. Whether that is the right trade is the last
question.

## 4. Where should the alarm go, and would it catch North Korea's explosions?

`predict` flags an event when its probability is above 0.5. Nothing says 0.5 is right. A missed
explosion is a secret test that got through; a false alarm accuses a state that did nothing, and
an inspection team can check a flagged event by other means. The **alarm level** is that choice:
the probability at or above which you flag an event. Lower it and you miss fewer explosions but
accuse more earthquakes.

Keep the scale in mind too. One event in ten is an explosion here only because these regions were
chosen around test sites. In a world catalogue explosions are far rarer, so even a small share of
earthquakes falsely flagged would mean many flags for every real explosion.

### ✏️ Your turn 6

**What happens when the alarm is lowered to 0.1?**

**Why:** at 0.5 the model missed 5 of 35 explosions, and
each miss is a test that got through. A lower level should catch more; count what it costs.

1. **`prob_explosion`**: `model.predict_proba(...)[:, 1]` on the held-out events' two columns: one
   probability per event (`[:, 1]` is the column for True, an explosion).
2. **`guess_low_alarm`**: True where `prob_explosion` is 0.1 or more.
3. **`counts_low_alarm`**: `confusion_matrix` of the held-out label and `guess_low_alarm`. The
   lines that read the hits and false alarms out of it are given.

**Use these names**, because the self-check looks for them: `prob_explosion` and
`counts_low_alarm`.

In [ ]:
# ← your answer here: replace every ... with code
prob_explosion = model.predict_proba(held_out[[..., ...]])[:, 1]  # the held-out events' two magnitudes
guess_low_alarm = prob_explosion >= ...                           # the lower alarm level
counts_low_alarm = confusion_matrix(..., ...)                     # the held-out label, then the guess at the lower alarm
print(counts_low_alarm)
print("alarm 0.1:", counts_low_alarm[1, 1], "explosions flagged and", counts_low_alarm[0, 1], "earthquakes flagged")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert counts_low_alarm[1, 1] == 34 and counts_low_alarm[0, 1] == 17, "prob_explosion is predict_proba(...)[:, 1] on held_out[['mb', 'Ms']]; flag at 0.1 or more"
print(f"✓ Your turn 6 — at alarm 0.1: {counts_low_alarm[1, 1]} explosions and {counts_low_alarm[0, 1]} earthquakes flagged")

At 0.5: 30 of 35 explosions, and no earthquake. At
0.1: 34 of 35, and 17
earthquakes. Every level makes its own trade, and an **ROC curve** draws them all at once: the
share of explosions caught against the share of earthquakes falsely flagged, one point per level.
The closer the curve hugs the top left, the better the model.

One point is already known. At 0.5 the model caught 30 of 35
explosions, a share of 0.857, and flagged 0 of
301 earthquakes, a share of 0: the point (0, 0.857).
The share caught is the recall at that level. A rule that flagged events at random would catch the
same share of each label, so it would sit on the diagonal: that is the coin toss.

### ✏️ Your turn 7

**What does every alarm level trade?**

**Why:** one level is one point. The whole curve shows how good the model is before anyone chooses
a level.

1. **The loop**: for each level in `levels` (given, from 1 down to 0), flag the held-out events
   whose probability is at or above it; append to `hit_rates` the share of the explosions flagged
   (hits over explosions) and to `false_alarm_rates` the share of the earthquakes flagged.
2. **The figure**: `hit_rates` up, `false_alarm_rates` across, with the diagonal a coin toss would
   give.
3. **`auc`**: `roc_auc_score(is_explosion, prob_explosion)`, the area under the curve: 1 for a
   perfect model, 0.5 for guessing. It is the chance that a random explosion gets a higher
   probability than a random earthquake, and it uses every possible level, not only the eleven
   dots.

**Use these names**, because the self-check looks for them: `hit_rates`, `false_alarm_rates` and
`auc`.

In [ ]:
# ← your answer here: replace every ... with code
levels = [1.0, 0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.0]
is_explosion = held_out["is_explosion"]
hit_rates = []
false_alarm_rates = []
for level in levels:
    flagged = prob_explosion >= ...                                            # this alarm level
    hit_rates.append((flagged & is_explosion).sum() / ...)                     # over the number of held-out explosions
    false_alarm_rates.append((flagged & (is_explosion == False)).sum() / ...)  # over the number of held-out earthquakes
auc = roc_auc_score(..., ...)                                                  # the held-out label, then the probabilities
print("area under the ROC curve:", round(auc, 3))

plt.figure(figsize=(5.5, 5.5))
plt.plot(..., ..., marker="o", color="black", label="the fitted line")  # false-alarm share across, hit share up
plt.plot([0, 1], [0, 1], color="0.6", ls="--", label="a coin toss")
plt.xlabel("share of earthquakes falsely flagged")
plt.ylabel("share of explosions caught")
plt.title("ROC curve on the held-out events")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert [round(x, 3) for x in hit_rates] == [0.0, 0.429, 0.686, 0.857, 0.943, 0.971, 0.971, 1.0, 1.0, 1.0, 1.0], "for each level: flagged = prob_explosion >= level; hits over the number of explosions"
assert [round(x, 3) for x in false_alarm_rates] == [0.0, 0.0, 0.0, 0.0, 0.0, 0.013, 0.056, 0.163, 0.329, 0.532, 1.0], "flagged earthquakes over the number of earthquakes"
assert round(auc, 3) == 0.995, "roc_auc_score(held_out['is_explosion'], prob_explosion)"
print(f"✓ Your turn 7 — area under the ROC curve: {auc:.3f}")

The curve climbs almost straight up: the model catches most explosions before it flags a single
earthquake, and the area under it is 0.995. The area judges how well the model orders the
events, not where any one alarm level falls. Every point on the curve is one possible rule. Which one
to use is not a question the data can answer; it depends on what a miss and a false accusation
each cost.

**North Korea.** The line was fitted on Nevada, Central Asia and Novaya Zemlya, 1970 to 1996.
North Korea has tested since 2006 at a site none of those events came from. The ISC lists five of
its explosions with both magnitudes: a fair test of the line at a place it never saw.

### ✏️ Your turn 8

**Would the line have caught North Korea's explosions?**

**Why:** a model is only as useful as it is on the events it will meet next.

1. **`prob_korea`**: the model's probability for each row of `korea`.
2. **`flagged_05`** and **`flagged_low`**: how many of the five it flags at 0.5 and at 0.1.
   The last line, given, counts how many the line by eye of Your turn 4 flags, for comparison.

**Use these names**, because the self-check looks for them: `prob_korea`, `flagged_05` and
`flagged_low`.

In [ ]:
# ← your answer here: replace every ... with code
prob_korea = model.predict_proba(korea[[..., ...]])[:, 1]  # North Korea's two magnitudes
flagged_05 = (prob_korea >= ...).sum()                     # the usual alarm level
flagged_low = (prob_korea >= ...).sum()                    # the lower alarm level
for date, prob in zip(korea["date"], prob_korea):
    print(date, " probability of an explosion", round(prob, 2))
print("flagged at 0.5:", flagged_05, "of 5;  at 0.1:", flagged_low, "of 5")
flagged_eye = (korea["Ms"] < korea["mb"] - 0.95).sum()     # the line by eye of Your turn 4
print("the line by eye flags", flagged_eye, "of 5")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert flagged_05 == 1 and flagged_low == 4, "prob_korea is predict_proba(korea[['mb', 'Ms']])[:, 1]; count those at or above each level"
print(f"✓ Your turn 8 — North Korea: {flagged_05} of 5 flagged at 0.5, {flagged_low} of 5 at 0.1")

At 0.5 the model flags only the largest explosion, 2017's (probability 0.88). The
other four get 0.06, 0.24, 0.18 and 0.11. Yet the plain line by eye,
with its slope of 1, flags 3 of the 5.

**The slope is the reason.** North Korea's explosions are small: mb 4.8 to
5.3, but for 2017's 6.3. Their Ms sits 0.7
to 1.1 below their mb, as it does for the small explosions in the table (a median
of 1.0 below). At mb 5.1 the fitted boundary flags an event only if its Ms is
more than 1.43 below its mb; at mb 6.3, more than 0.56.
The slope was learned from large explosions, and it does not carry down to small ones. A model
that scores better on events like its training events can still do worse on new ones.

**What monitoring agencies use.** They faced the same question. The screening line used by the treaty
organisation's data centre until 2010, Ms = 1.25 mb − 2.2, has a slope between the two; with this
table's magnitudes it flags 3 of the five.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week09/mb_ms_screening_line.png" width="440">

*The world's earthquakes of 2008 (grey) and six explosions, with the screening line (dashed).
North Korea's first two tests, 2006 and 2009, sit right on the line. These magnitudes come from
the treaty organisation's own network, so they differ from this table's by a few tenths. Figure:
Selby et al. (2012), shown in G. Ekström's "Seismology in MTV" slides (Columbia University, 2019).*

The map below shows where the five fall against the fitted model and the line by eye. Lowered to
0.1, the alarm catches four of the five, at the cost on the held-out events of
17 false alarms. One seismic measure is never enough: monitoring agencies
use several together, and a flag starts an investigation rather than ending one.

In [ ]:
# the model's probability at every point of grid, the table of mb-Ms points made in the setup cell
grid_probability = model.predict_proba(grid[["mb", "Ms"]])[:, 1]

plt.figure(figsize=(7, 5.5))
plt.scatter(grid["mb"], grid["Ms"], c=grid_probability, cmap="coolwarm", s=12, marker="s")
plt.colorbar(label="probability of an explosion")
plt.scatter(held_earthquakes["mb"], held_earthquakes["Ms"], s=8, color="white", edgecolor="0.3", label="held-out earthquake")
plt.scatter(held_explosions["mb"], held_explosions["Ms"], s=14, color="black", label="held-out explosion")
plt.scatter(korea["mb"], korea["Ms"], s=150, marker="*", color="yellow", edgecolor="black", label="North Korea")
plt.plot(mb_ends, mb_ends - 0.95, color="black", ls="--", label="line by eye")
plt.ylim(2.5, 8)
plt.xlabel("body-wave magnitude mb")
plt.ylabel("surface-wave magnitude Ms")
plt.title("The model's probability everywhere, with North Korea's explosions")
plt.legend(loc="upper left")
plt.show()

## The question, answered

Mostly, with two magnitudes and an honest score. "Always earthquake" is right 0.895
of the time and finds nothing; a cut on mb finds explosions with many false alarms (F1
0.554); a depth of 0 looks perfect only because the analyst already knew. A straight line
through mb and Ms, fitted on some events and checked on others, finds 30 of
35 held-out explosions with no false alarm. Where to set the alarm is a choice
the ROC curve shows but cannot make. And at North Korea's small explosions, the fitted line misses
what the plainer line by eye catches: its slope was learned from large explosions. A better score
on familiar events does not promise a better answer on new ones.

## Week 9 summary

### The science

**Earthquake or nuclear explosion?**  
When one label is rare, accuracy hides the misses: 'always earthquake' scores 0.895 and finds no explosion; precision, recall and F1 count the two kinds of mistake apart. Logistic regression draws a straight line through two columns and gives every event a probability; checked on events it never saw, the line through mb and Ms finds explosions that each magnitude alone misses. But its slope was learned from large explosions: at North Korea's small ones the plain line by eye does better. Where to set the alarm is a choice between missed explosions and false accusations; the ROC curve shows every choice at once, and a column that already knows the answer (a depth the analyst set to 0) is leakage, not skill.

- **Logistic regression.** The same straight line — but now it outputs a probability between 0 and 1.
- **Precision / recall.** Of the ones you flagged, how many were right? Of the real ones, how many did you catch?
- **Leakage.** You got 99 percent. Be suspicious. Did one of your columns already know the answer?
- **Baseline.** Write the dumbest rule you can, first. Any model that cannot beat it is decoration.
- **Body-wave and surface-wave magnitude.** Two sizes for one event: mb is read from the first, fast P wave, Ms from the slow surface waves that follow. An explosion pushes out from one small spot and makes weak surface waves for its mb; an earthquake slips along a fault and makes strong ones.
- **Hit, miss, false alarm.** A hit is an explosion you flagged, a miss an explosion you did not, a false alarm an earthquake you flagged.
- **Alarm level.** The probability at or above which you flag an event. Lower it and you miss fewer explosions but accuse more earthquakes.
- **ROC curve.** Every alarm level at once: the share of explosions caught against the share of earthquakes falsely flagged. The closer the curve hugs the top left, the better the model.
- **Decision boundary.** Where the model's answer flips from one label to the other — for logistic regression, always a straight line.

### The code

| Function | What it does |
|---|---|
| **Python** | |
| `list.append(x)` | add one item to the end of a list |
| `len(table)` | how many rows |
| `for x in things:` | do the same thing once for each item |
| `round(x, 1)` | trim a long decimal — 9.1 rather than sixteen digits |
| **NumPy** | |
| `mask & other_mask` | True only where both masks are True, the array version of and |
| `np.array(list)` | the container that does arithmetic to every number at once |
| **pandas** | |
| `mask.mean()` | the proportion of a mask that is True, in one line |
| **Matplotlib** | |
| `plt.plot(x, y)` | join points with a line — here, the coastline |
| `plt.scatter(x, y)` | a dot per pair of values |
| `plt.hist(x, bins=n)` | how one list of numbers is spread out |
| `plt.axvline(x)` | a vertical line at x, to mark a boundary on a plot |
| **scikit-learn** | |
| `LogisticRegression()` | draw the straight line that best separates two labelled groups |
| `accuracy_score(y, pred)` | the fraction it got right — useless when one label is rare |
| `precision_score(y, pred)` | of the ones it flagged, how many really were explosions |
| `recall_score(y, pred)` | of the explosions there were, how many it flagged |
| `f1_score(y, pred)` | precision and recall in one number, so two models can be compared |
| `train_test_split(table, test_size=0.3, random_state=n, stratify=column)` | cut the rows into a training part and a held-out part, keeping the share of each label |
| `model.fit(X, y) / model.predict(X)` | learn from the training rows, then predict for any rows |
| `confusion_matrix(y, pred)` | the two kinds of mistake counted apart: misses in one corner, false alarms in the other |
| `model.predict_proba(X)[:, 1]` | the model's probability that each row is the True label |
| `roc_auc_score(y, prob)` | one number for the whole ROC curve: 1 is perfect, 0.5 is guessing |

## Homework

Class fitted the line on events from all three regions. A monitoring agency rarely has that luxury:
it learns from the tests it knows and must watch places it has never recorded. Here you fit the
line on the western US alone (405 events, 36 explosions, almost all
at the Nevada Test Site) and check it on Eurasia (Central Asia and Novaya Zemlya together:
713 events, 81 explosions). Each part is a step from class again, with less
written in.

The two tables come from `events`, which the setup cell makes, so nothing from class is needed if
you restart: run the setup cell, then the next cell.

In [ ]:
west_us = events[events["region"] == "western US"]
eurasia = events[events["region"] != "western US"]
print(len(west_us), "events in the western US;", len(eurasia), "in Eurasia")

### ✏️ Your turn 9

**In Eurasia, what do "always earthquake" and a cut on mb score?**

**Why:** a model is only worth something in Eurasia if it beats the rules that need no fitting there.
These are the baselines of Your turns 1 and 2, in the new place.

1. **`eurasia_accuracy`**: the accuracy of a guess of False for every Eurasian event.
2. **`eurasia_f1_mb`**: the F1 of the cut at mb 5.7 on the Eurasian events.

**Use these names**, because the self-check looks for them: `eurasia_accuracy` and
`eurasia_f1_mb`.

In [ ]:
# ← your answer here: replace every ... with code
eurasia_label = eurasia["is_explosion"]
eurasia_accuracy = accuracy_score(..., eurasia["mb"] > 100)  # the Eurasian label; the guess of False is given
eurasia_f1_mb = f1_score(..., eurasia["mb"] >= ...)          # the Eurasian label; the cut of class
print("always earthquake: accuracy", round(eurasia_accuracy, 3), "  cut on mb: F1", round(eurasia_f1_mb, 3))
print(confusion_matrix(eurasia_label, eurasia["mb"] >= 5.7))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(eurasia_accuracy, 3) == 0.886, "a guess of False for every Eurasian event, e.g. eurasia['mb'] > 100"
assert round(eurasia_f1_mb, 3) == 0.667, "the cut eurasia['mb'] >= 5.7"
print(f"✓ Your turn 9 — Eurasia: always earthquake {eurasia_accuracy:.3f}, cut on mb F1 {eurasia_f1_mb:.3f}")

**What the numbers mean.** How does each baseline compare with class? Which mistake does the cut on mb make most?

### ✏️ Your turn 10

**Does a line fitted in the western US find the explosions of Eurasia?**

**Why:** an agency learns from the tests it knows and must watch places it has never recorded. So
the line learns in one place and is scored in another, as in Your turn 5, with the two tables as the
split.

1. **`model_west`**: logistic regression fitted on `west_us`: `mb` and `Ms`, then the label.
2. **`guess_eurasia`**: its guess for every Eurasian event.
3. **`eurasia_counts`** and **`eurasia_f1_model`**: the confusion matrix and the F1 of that guess.

**Use these names**, because the self-check looks for them: `model_west`, `eurasia_counts` and
`eurasia_f1_model`.

In [ ]:
# ← your answer here: replace every ... with code
model_west = LogisticRegression().fit(west_us[[..., ...]], west_us[...])  # the two magnitudes, then the label
guess_eurasia = model_west.predict(eurasia[[..., ...]])                   # Eurasia's two magnitudes
eurasia_counts = confusion_matrix(..., ...)                               # the Eurasian label, then the guess
eurasia_f1_model = f1_score(..., ...)                                     # the same two
print(eurasia_counts)
print("fitted in the western US, checked in Eurasia: F1", round(eurasia_f1_model, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert eurasia_counts[1, 1] == 70 and eurasia_counts[0, 1] == 0, "fit on west_us[['mb', 'Ms']], predict on eurasia[['mb', 'Ms']]"
assert round(eurasia_f1_model, 3) == 0.927, "f1_score of the Eurasian label and the guess"
print(f"✓ Your turn 10 — fitted in the western US, checked in Eurasia: F1 {eurasia_f1_model:.3f}")

**What the numbers mean.** Compare the fitted line with the cut on mb of Your turn 9. Which mistake did fitting remove, and
which one is left?

In [ ]:
# the western model's probability at every point of grid, made in the setup cell
grid_probability = model_west.predict_proba(grid[["mb", "Ms"]])[:, 1]
eurasia_explosions = eurasia[eurasia_label]
eurasia_earthquakes = eurasia[eurasia_label == False]

plt.figure(figsize=(7, 5.5))
plt.scatter(grid["mb"], grid["Ms"], c=grid_probability, cmap="coolwarm", s=12, marker="s")
plt.colorbar(label="probability of an explosion (model fitted in the western US)")
plt.scatter(eurasia_earthquakes["mb"], eurasia_earthquakes["Ms"], s=8, color="white", edgecolor="0.3", label="earthquake")
plt.scatter(eurasia_explosions["mb"], eurasia_explosions["Ms"], s=14, color="black", label="explosion")
plt.xlabel("body-wave magnitude mb")
plt.ylabel("surface-wave magnitude Ms")
plt.title("Eurasia on the western model's map")
plt.legend(loc="upper left")
plt.show()

### ✏️ Your turn 11

**Where would you set the alarm for Eurasia?**

**Why:** an agency watching Eurasia must pick a level before it sees the next event. Count both
mistakes at 0.5 and at 0.2 to see what the choice costs.

1. **`prob_eurasia`**: `model_west`'s probability for every Eurasian event.
2. **`counts_05`** and **`counts_02`**: the confusion matrices when flagging at 0.5 or more and at
   0.2 or more.

**Use these names**, because the self-check looks for them: `prob_eurasia`, `counts_05` and
`counts_02`.

In [ ]:
# ← your answer here: replace every ... with code
prob_eurasia = model_west.predict_proba(eurasia[[..., ...]])[:, 1]  # Eurasia's two magnitudes
counts_05 = confusion_matrix(..., prob_eurasia >= ...)              # the Eurasian label; the usual alarm level
counts_02 = confusion_matrix(..., prob_eurasia >= ...)              # the same, at the lower level
print("alarm 0.5:", counts_05[1, 0], "misses,", counts_05[0, 1], "false alarms")
print("alarm 0.2:", counts_02[1, 0], "misses,", counts_02[0, 1], "false alarms")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (counts_05[1, 0], counts_05[0, 1]) == (11, 0), "flag where prob_eurasia >= 0.5"
assert (counts_02[1, 0], counts_02[0, 1]) == (2, 15), "flag where prob_eurasia >= 0.2"
print(f"✓ Your turn 11 — alarm 0.2: {counts_02[1, 0]} misses and {counts_02[0, 1]} false alarms")

**What the numbers mean.** Which level would you recommend to a monitoring agency, and what would you tell it about the cost?

### ✏️ Your turn 12

**How good is the western line in Eurasia at every alarm level?**

**Why:** the alarm levels of Your turn 11 are two points; the whole curve says how well the
western model orders Eurasia's events before anyone picks a level.

1. **The loop**: for each level in `levels`, append to `eurasia_hit_rates` and
   `eurasia_false_alarm_rates`, as in class.
2. **`eurasia_auc`**: the area under the curve.
3. **The figure**: the curve and the diagonal.

**Use these names**, because the self-check looks for them: `eurasia_hit_rates`,
`eurasia_false_alarm_rates` and `eurasia_auc`.

In [ ]:
# ← your answer here: replace every ... with code
levels = [1.0, 0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.0]
eurasia_hit_rates = []
eurasia_false_alarm_rates = []
for level in levels:
    flagged = prob_eurasia >= ...                                    # this alarm level
    eurasia_hit_rates.append((flagged & eurasia_label).sum() / ...)  # over the number of Eurasian explosions
    eurasia_false_alarm_rates.append((flagged & (eurasia_label == False)).sum() / ...)  # over the number of Eurasian earthquakes
eurasia_auc = roc_auc_score(..., ...)                                # the Eurasian label, then the probabilities
print("area under the ROC curve in Eurasia:", round(eurasia_auc, 3))

plt.figure(figsize=(5.5, 5.5))
plt.plot(..., ..., marker="o", color="black", label="line fitted in the western US")  # false-alarm shares across, hit shares up
plt.plot([0, 1], [0, 1], color="0.6", ls="--", label="a coin toss")
plt.xlabel("share of earthquakes falsely flagged")
plt.ylabel("share of explosions caught")
plt.title("ROC curve in Eurasia")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert [round(x, 3) for x in eurasia_hit_rates] == [0.0, 0.247, 0.728, 0.864, 0.938, 0.975, 0.988, 0.988, 1.0, 1.0, 1.0], "for each level: flagged = prob_eurasia >= level; hits over explosions"
assert [round(x, 3) for x in eurasia_false_alarm_rates] == [0.0, 0.0, 0.0, 0.0, 0.009, 0.024, 0.097, 0.288, 0.616, 0.812, 1.0], "flagged earthquakes over earthquakes"
assert round(eurasia_auc, 3) == 0.993, "roc_auc_score(eurasia_label, prob_eurasia)"
print(f"✓ Your turn 12 — area under the ROC curve in Eurasia: {eurasia_auc:.3f}")

**What the numbers mean.** Compare the area with class's (0.995). If the curve is nearly as good, why did the line still
miss explosions at 0.5?